# What the Grid Actually Faces When These Forecasts Come True

## Objectives

Every earlier chapter in this part treated a forecast as the end product:
a number, an interval, a ranking. Part 4 already built a real OpenDSS
network model of this exact 342-customer AusNet feeder and used it to run
hosting-capacity studies, but always against a *historical* day already on
record, never a forecast. This chapter closes that gap directly: take
Chapter 4's own zero-shot forecast, seriously, and use it to drive Part 4's
own network model instead of replaying a day that already happened.

The real question is not whether the network model works, Part 4 already
proved that. It is whether relying on a forecast changes the answer to a
real operational question a {{< acr DSO >}} actually asks: at what {{< acr PV >}}
penetration level does this feeder start breaching its own voltage limit,
and does trusting a point forecast alone understate that risk compared to
taking the forecast's own uncertainty seriously.

By the end you will be able to:

- Reuse Part 4's own `Circuit` and hosting-capacity sweep machinery
  unmodified, parameterized on a forecasted load trajectory instead of a
  historical one.
- Compare a historical replay, a zero-shot point forecast, and a zero-shot
  upper-quantile stress scenario, run through the identical network at the
  identical {{< acr PV >}} penetration levels.
- Check whether either of Part 4's own rule-based mitigation levers,
  Volt-Watt or Volt-VAr, fixes a violation the forecast-driven scenario
  reveals but the historical replay does not.


## Setup: the same network, the same population, a forecast instead of a replay

Same real AusNet 342-customer pool and the same OpenDSS feeder model Part 4
Chapter 2 built and validated, `Circuit.load` against the identical
`LVcircuit-master.txt`. Nothing is re-derived: `build_ausnet_network()`
below is the exact same function, byte for byte.

In [ ]:
from pathlib import Path
import warnings

from great_tables import GT, md
from lets_plot import (
    LetsPlot,
    aes,
    element_text,
    geom_bar,
    geom_hline,
    geom_line,
    geom_point,
    geom_rect,
    ggplot,
    ggsize,
    labs,
    scale_color_manual,
    scale_fill_manual,
    theme,
)
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

from ark.dss.circuit import Circuit
from ark.dss.mitigation import apply_volt_var, apply_volt_watt
from ark.plot.gt_style import themed_gt
from ark.plot.theme import modern_theme
from ark.plot.tokens import AI_ACCENT, INFO, PRIMARY, SUCCESS, WARNING

LetsPlot.setup_html(isolated_frame=True)
UNBOLD_AXES = theme(axis_title_x=element_text(face="plain"), axis_title_y=element_text(face="plain"))
FULL_WIDTH = 650

DATA_DIR = Path("../../resources/cvar_flexibility/data/timeseries-lv")
STEPS_PER_DAY = 48
TRANSFORMER_KVA = 500.0
LOW, HIGH = 0.94, 1.10  # same ANSI-style compliance band as ark.dss.violations' own default
DAY_IDX = 363  # the sunniest real day in the dataset, Part 4 Chapter 2's own hosting-capacity convention
RNG_SEED = 42

SCENARIO_COLORS = {
    "Historical replay": PRIMARY,
    "Zero-shot point forecast": INFO,
    "Zero-shot P95 stress": AI_ACCENT,
}

load_data = np.load(DATA_DIR / "Residential load data 30-min resolution.npy")
pv_data = np.load(DATA_DIR / "Residential PV data 30-min resolution.npy")
n_customers = load_data.shape[0]
print(f"load_data: {load_data.shape} (customers, days, half-hours)")


def build_ausnet_network() -> Circuit:
    circuit = Circuit.load(DATA_DIR / "LVcircuit-master.txt", solve=False)
    circuit.command("Set VoltageBases = [22.0, 0.400]")
    circuit.command("calcvoltagebases")
    return circuit


circuit = build_ausnet_network()
circuit.solve()
circuit.summary()

load_data: (342, 365, 48) (customers, days, half-hours)


{'converged': True,
 'n_buses': 61,
 'n_lines': 59,
 'n_loads': 31,
 'n_transformers': 1,
 'total_p_loss_kw': 0.6107992120127365,
 'total_q_loss_kvar': 0.15519491021295995}

## Chapter 4's own zero-shot forecast, now driving a network instead of being scored

Chronos-2, recomputed exactly as Chapter 4 built it, no fitting step on
this data. Each customer's own day 362 becomes the context window; day 363,
the same sunniest real day Part 4 Chapter 2's own hosting-capacity study
already uses, becomes the forecast target. Chronos-2 hands back both a
point forecast and a full quantile spread, so this section keeps two real
scenarios from the same single forecast call: the point forecast (`loc`),
and the 0.95 quantile, a real upper-demand scenario a {{< acr DSO >}}
planning only off the point forecast would never see. Whether that upper
quantile is actually the *worse* scenario is not assumed here; the sweep
below checks it directly, and the real answer turns out to depend on which
constraint and which {{< acr PV >}} penetration level are in question.


In [ ]:
from twiga.models.foundational.chronos2_model import Chronos2Config, Chronos2Model

X_fm = load_data[:, DAY_IDX - 1, :][:, :, None].astype(np.float32)
chronos_model = Chronos2Model(Chronos2Config(horizon=STEPS_PER_DAY))
chronos_out = chronos_model.forecast(X_fm)

loc_forecast = chronos_out["loc"].squeeze(-1)  # (n_customers, 48), point forecast
p95_idx = chronos_out["quantile_levels"].index(0.95)
p95_forecast = chronos_out["quantiles"][:, p95_idx, :, 0]  # (n_customers, 48), upper-demand scenario

actual_day = load_data[:, DAY_IDX, :]  # (n_customers, 48), what actually happened, the historical replay
forecast_mae = np.abs(actual_day - loc_forecast).mean()
p95_gap = (p95_forecast - loc_forecast).mean()
print(f"zero-shot point forecast, day {DAY_IDX}, population MAE: {forecast_mae:.4f}")
print(f"P95 scenario sits above the point forecast by, on average, {p95_gap:.4f} kW per half-hour")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

zero-shot point forecast, day 363, population MAE: 0.2135
P95 scenario sits above the point forecast by, on average, 0.8714 kW per half-hour


## One sweep function, three real load sources

`run_scenario` below is Part 4 Chapter 2's own `run_penetration`, generalized
in exactly one way: instead of indexing a fixed historical day out of
`load_data`, it takes a `load_source` array directly, one row per customer.
Every other mechanic, the customer-to-PV assignment, the PV profile, the
reactive-power synthesis, the mitigation control, the per-step voltage and
transformer-utilization readout, is unchanged from Part 4's own function.


In [ ]:
def run_scenario(
    load_source: np.ndarray,
    penetration: int,
    *,
    pv_kva: float = 5.0,
    control: str | None = None,
    seed: int = RNG_SEED,
) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    circuit = build_ausnet_network()
    loads = list(circuit.loads)

    customer_indices = rng.choice(n_customers, size=len(loads), replace=False)
    n_with_pv = round(len(loads) * penetration / 100)
    pv_customers = set(rng.choice([load.name for load in loads], size=n_with_pv, replace=False))
    pv_profile = pv_data[DAY_IDX, :]

    for load, customer_idx in zip(loads, customer_indices, strict=True):
        p = load_source[customer_idx]
        pf = rng.uniform(0.95, 0.99, len(p))
        q = np.sqrt(np.maximum((p / pf) ** 2 - p**2, 0))
        circuit.command(
            f"New LoadShape.profile_{load.name} npts=48 minterval=30 useactual=1 pmult={p.tolist()} qmult={q.tolist()}"
        )
        circuit.command(f"edit load.{load.name} daily=profile_{load.name}")

        if load.name in pv_customers:
            circuit.command(
                f"new PVSystem.pv_{load.name} bus1={load.bus1} phases=1 kva={pv_kva} pmpp={pv_kva} "
                "pf=1 kV=(0.4 3 sqrt /) model=1 irradiance=1 %cutin=0.05 %cutout=0.05"
            )
            circuit.command(
                f"New LoadShape.pv_profile_{load.name} npts=48 minterval=30 useactual=1 pmult={pv_profile.tolist()}"
            )
            circuit.command(f"edit PVSystem.pv_{load.name} daily=pv_profile_{load.name}")

    if control == "voltwatt":
        apply_volt_watt(circuit)
    elif control == "voltvar":
        apply_volt_var(circuit)

    rows = []
    for step in circuit.solve_daily(steps=STEPS_PER_DAY):
        voltages = circuit.bus_voltages().query("bus != @circuit.source_bus")
        transformer_power = circuit.element_powers("transformers")
        s_kva = np.sqrt(transformer_power["p_kw"] ** 2 + transformer_power["q_kvar"] ** 2).sum()
        rows.append(
            {
                "step": step,
                "penetration": penetration,
                "vmax_pu": voltages["vmag_pu"].max(),
                "vmin_pu": voltages["vmag_pu"].min(),
                "utilization_pct": s_kva / TRANSFORMER_KVA * 100,
            }
        )
    return pd.DataFrame(rows)

## Does trusting the forecast change the hosting-capacity answer?

Three real load sources, the same network, the same {{< acr PV >}} penetration
sweep, the same customer-to-PV assignment (same `seed`, so any difference
below comes from the load source alone, not from a different random draw of
which customers get PV): the historical replay Part 4 Chapter 2 already
studied, Chronos-2's zero-shot point forecast, and Chronos-2's own 0.95
upper-quantile scenario.


In [ ]:
PENETRATION_LEVELS = [0, 20, 40, 60, 80, 100]
SCENARIOS = {
    "Historical replay": actual_day,
    "Zero-shot point forecast": loc_forecast,
    "Zero-shot P95 stress": p95_forecast,
}

scenario_rows = []
for label, source in SCENARIOS.items():
    for pen in PENETRATION_LEVELS:
        result = run_scenario(source, pen)
        scenario_rows.append(
            {
                "scenario": label,
                "penetration": pen,
                "vmax_pu": result["vmax_pu"].max(),
                "utilization_pct": result["utilization_pct"].max(),
            }
        )
scenario_df = pd.DataFrame(scenario_rows)
scenario_df.pivot(index="penetration", columns="scenario", values="vmax_pu").round(4)

scenario,Historical replay,Zero-shot P95 stress,Zero-shot point forecast
penetration,,,
0,1.0823,1.0820,1.0823
20,1.0876,1.0859,1.0878
40,1.0956,1.0940,1.0959
60,1.0973,1.0929,1.0966
80,1.1047,1.1005,1.1040
100,1.1046,1.1004,1.1039


In [ ]:
(
    ggplot()
    + geom_rect(
        aes(xmin="xmin", xmax="xmax"),
        data=pd.DataFrame({"xmin": [0], "xmax": [100]}),
        ymin=LOW,
        ymax=HIGH,
        fill=SCENARIO_COLORS["Historical replay"],
        alpha=0.06,
        inherit_aes=False,
    )
    + geom_hline(yintercept=HIGH, linetype="dashed", color=WARNING, size=0.8)
    + geom_line(aes(x="penetration", y="vmax_pu", color="scenario"), data=scenario_df, size=1)
    + geom_point(aes(x="penetration", y="vmax_pu", color="scenario"), data=scenario_df, size=3)
    + scale_color_manual(values=SCENARIO_COLORS)
    + labs(
        x="PV penetration (% of customers)",
        y="Worst-case feeder voltage (pu)",
        title="Does a Forecast Change Which Penetration Level Breaches 1.10 pu?",
        color="",
    )
    + modern_theme(legend_pos="bottom")
    + UNBOLD_AXES
    + ggsize(FULL_WIDTH, 380)
)

A single number settles the first real question this section opened with:
the lowest {{< acr PV >}} penetration level at which each scenario's own
worst-case voltage first crosses the 1.10 pu compliance limit.


In [ ]:
def first_breach(df: pd.DataFrame, scenario: str) -> int | None:
    sub = df[df["scenario"] == scenario].sort_values("penetration")
    breach = sub[sub["vmax_pu"] > HIGH]
    return int(breach["penetration"].min()) if len(breach) else None


breach_summary = pd.DataFrame(
    [{"Scenario": label, "First breach at penetration": first_breach(scenario_df, label)} for label in SCENARIOS]
)
themed_gt(
    GT(breach_summary)
    .tab_header(title=md("**At What PV Penetration Does This Feeder First Breach 1.10 pu?**"))
    .tab_source_note(f"Day {DAY_IDX}, the sunniest real day, same customer-to-PV assignment across scenarios"),
    n_rows=len(breach_summary),
)

GT(_tbl_data=                   Scenario  First breach at penetration
0         Historical replay                           80
1  Zero-shot point forecast                           80
2      Zero-shot P95 stress                           80, _body=<great_tables._gt_data.Body object at 0x13d0a41d0>, _boxhead=Boxhead([ColInfo(var='Scenario', type=<ColInfoTypeEnum.default: 1>, column_label='Scenario', column_align='left', column_width=None), ColInfo(var='First breach at penetration', type=<ColInfoTypeEnum.default: 1>, column_label='First breach at penetration', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x13d0e2ff0>, _spanners=Spanners([]), _heading=Heading(title=Md(text='**At What PV Penetration Does This Feeder First Breach 1.10 pu?**'), subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x13d0a4470>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x13cf2faa0>, _source_notes=['Day 363, the sunniest real day, same customer-to-PV assignment across scenarios'], _footnotes=[], _styles=[StyleInfo(locname=LocTitle(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#171717', font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocSubTitle(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#6B7280', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='Scenario', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font='Jost', size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='First breach at penetration', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font='Jost', size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocSourceNotes(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#6B7280', font=None, size=None, align=None, v_align=None, style='italic', weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocBody(columns=None, rows=[1], mask=None), grpname=None, colname='Scenario', rownum=1, colnum=None, styles=[CellStyleFill(color='#EAF3FA')]), StyleInfo(locname=LocBody(columns=None, rows=[1], mask=None), grpname=None, colname='First breach at penetration', rownum=1, colnum=None, styles=[CellStyleFill(color='#EAF3FA')])], _locale=<great_tables._gt_data.Locale object at 0x13cf2fb30>, _formats=[], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='100%'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['Libre Franklin', 'Segoe UI', 'Roboto', 'Helvetica Neue', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='13px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), 

All three scenarios first breach at the same penetration level here, so the
forecast does not move *this* particular answer. A second, more useful
finding sits underneath that headline number: does the upper-quantile
scenario actually behave like a uniformly worse case, the way "P95" sounds
like it should?

In [ ]:
scenario_df.pivot(index="penetration", columns="scenario", values="utilization_pct").round(2)

scenario,Historical replay,Zero-shot P95 stress,Zero-shot point forecast
penetration,,,
0,4.00,9.30,2.81
20,4.01,9.11,3.87
40,10.03,8.92,9.90
60,16.88,11.64,16.75
80,22.82,17.62,22.69
100,28.67,23.40,28.53


In [ ]:
(
    ggplot(scenario_df, aes(x="penetration", y="utilization_pct", color="scenario"))
    + geom_line(size=1)
    + geom_point(size=3)
    + scale_color_manual(values=SCENARIO_COLORS)
    + labs(
        x="PV penetration (% of customers)",
        y="Transformer utilization (%)",
        title="Which Scenario Stresses the Transformer Most Depends on PV Penetration",
        color="",
    )
    + modern_theme(legend_pos="bottom")
    + UNBOLD_AXES
    + ggsize(FULL_WIDTH, 380)
)

No, and the real reason is physical, not statistical. At low {{< acr PV >}}
penetration the transformer's own throughput is mostly driven by load
itself, so the higher-demand scenario does raise utilization, exactly what
"P95 means worse" assumes. Past roughly 40% penetration, that reverses: a
customer with higher local demand consumes more of their own {{< acr PV >}}
export before it ever reaches the transformer, so higher demand *lowers*
both transformer utilization and worst-case voltage, the same real
compensatory mechanism Part 8 Chapter 1 already found between {{< acr PV >}}
and {{< acr EV >}} activity, now showing up between forecast demand and
{{< acr PV >}} export instead. A {{< acr DSO >}} that reflexively treats a
forecast's upper quantile as the universal worst case for every constraint
gets this backwards at high {{< acr PV >}} penetration; which quantile is
actually the risk depends on which constraint, and how much {{< acr PV >}} is
already on the feeder, not on "higher demand" as a blanket rule.

## Does a rule-based mitigation lever fix what the real worst case reveals?

Part 4 Chapter 2 already built and validated two real inverter-control
levers, Volt-Watt and Volt-VAr, reused here unmodified via
`ark.dss.mitigation`. Applied against whichever of the three scenarios
above is the *actual* worst case for voltage at the shared breach
penetration, not assumed in advance to be the upper-quantile one, since the
section above just found that assumption does not hold at this penetration
level.

In [ ]:
breach_penetration = min(p for p in (first_breach(scenario_df, label) for label in SCENARIOS) if p is not None)

worst_case_row = (
    scenario_df[scenario_df["penetration"] == breach_penetration].sort_values("vmax_pu", ascending=False).iloc[0]
)
worst_case_label = worst_case_row["scenario"]
worst_case_source = SCENARIOS[worst_case_label]

mitigation_rows = []
for control_name, control in [("No control", None), ("Volt-Watt", "voltwatt"), ("Volt-VAr", "voltvar")]:
    result = run_scenario(worst_case_source, breach_penetration, control=control)
    mitigation_rows.append(
        {
            "Control": control_name,
            "vmax_pu": round(result["vmax_pu"].max(), 4),
            "Still breaching 1.10 pu": bool(result["vmax_pu"].max() > HIGH),
        }
    )
mitigation_df = pd.DataFrame(mitigation_rows)

themed_gt(
    GT(mitigation_df)
    .tab_header(title=md("**Does Volt-Watt or Volt-VAr Fix the Real Worst Case?**"))
    .tab_source_note(f"PV penetration {breach_penetration}%, {worst_case_label.lower()} load, day {DAY_IDX}"),
    n_rows=len(mitigation_df),
)

GT(_tbl_data=      Control  vmax_pu  Still breaching 1.10 pu
0  No control   1.1047                     True
1   Volt-Watt   1.0878                    False
2    Volt-VAr   1.0710                    False, _body=<great_tables._gt_data.Body object at 0x13d0f28d0>, _boxhead=Boxhead([ColInfo(var='Control', type=<ColInfoTypeEnum.default: 1>, column_label='Control', column_align='left', column_width=None), ColInfo(var='vmax_pu', type=<ColInfoTypeEnum.default: 1>, column_label='vmax_pu', column_align='right', column_width=None), ColInfo(var='Still breaching 1.10 pu', type=<ColInfoTypeEnum.default: 1>, column_label='Still breaching 1.10 pu', column_align='center', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x13c865040>, _spanners=Spanners([]), _heading=Heading(title=Md(text='**Does Volt-Watt or Volt-VAr Fix the Real Worst Case?**'), subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x13cdb4c20>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x13cdb4e60>, _source_notes=['PV penetration 80%, historical replay load, day 363'], _footnotes=[], _styles=[StyleInfo(locname=LocTitle(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#171717', font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocSubTitle(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#6B7280', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='Control', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font='Jost', size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='vmax_pu', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font='Jost', size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='Still breaching 1.10 pu', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font='Jost', size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocSourceNotes(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#6B7280', font=None, size=None, align=None, v_align=None, style='italic', weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocBody(columns=None, rows=[1], mask=None), grpname=None, colname='Control', rownum=1, colnum=None, styles=[CellStyleFill(color='#EAF3FA')]), StyleInfo(locname=LocBody(columns=None, rows=[1], mask=None), grpname=None, colname='vmax_pu', rownum=1, colnum=None, styles=[CellStyleFill(color='#EAF3FA')]), StyleInfo(locname=LocBody(columns=None, rows=[1], mask=None), grpname=None, colname='Still breaching 1.10 pu', rownum=1, colnum=None, styles=[CellStyleFill(color='#EAF3FA')])], _locale=<great_tables._gt_data.Locale object at 0x13cdb6a20>, _formats=[], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='100%'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(

In [ ]:
CONTROL_COLORS = {"No control": AI_ACCENT, "Volt-Watt": WARNING, "Volt-VAr": SUCCESS}
mitigation_df["Control"] = pd.Categorical(mitigation_df["Control"], categories=CONTROL_COLORS.keys(), ordered=True)

(
    ggplot(mitigation_df, aes(x="Control", y="vmax_pu", fill="Control"))
    + geom_hline(yintercept=HIGH, linetype="dashed", color=WARNING, size=0.8)
    + geom_bar(stat="identity", width=0.5)
    + scale_fill_manual(values=CONTROL_COLORS)
    + labs(
        x="",
        y="Worst-case feeder voltage (pu)",
        title="Both Levers Bring the Real Worst Case Back Under 1.10 pu",
    )
    + modern_theme(legend_pos="none")
    + UNBOLD_AXES
    + ggsize(FULL_WIDTH, 380)
)